In [1]:
!pip -q install torch torchvision scikit-learn joblib kagglehub

In [1]:
import os, json, joblib, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import kagglehub


In [2]:
dataset_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset path:", dataset_path)

train_dir = os.path.join(dataset_path, "Training")
test_dir = os.path.join(dataset_path, "Testing")
assert os.path.exists(train_dir), f"Training folder not found: {train_dir}"
assert os.path.exists(test_dir), f"Testing folder not found: {test_dir}"



Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset path: /kaggle/input/brain-tumor-mri-dataset


In [3]:
train_tf = transforms.Compose([
transforms.Resize((224, 224)),
transforms.RandomHorizontalFlip(),
transforms.RandomRotation(8),
transforms.ToTensor(),
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
transforms.Resize((224, 224)),
transforms.ToTensor(),
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
test_ds = datasets.ImageFolder(test_dir, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Classes:", train_ds.classes)
print("Train samples:", len(train_ds), "Test samples:", len(test_ds))

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Train samples: 5600 Test samples: 1600


In [4]:
class BrainTumorNet(nn.Module):
  def __init__(self, num_classes=4):
    super().__init__()
    self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = self.backbone.classifier[1].in_features
    self.backbone.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.30),
    nn.Linear(in_features, num_classes)
    )

  def forward(self, x):
    return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainTumorNet(num_classes=len(train_ds.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 157MB/s]


In [5]:
epochs = 6
history = []

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

for xb, yb in train_loader:
  xb, yb = xb.to(device), yb.to(device)
  optimizer.zero_grad()
  logits = model(xb)
  loss = criterion(logits, yb)
  loss.backward()
  optimizer.step()
  running_loss += loss.item() * xb.size(0)

train_loss = running_loss / len(train_ds)

model.eval()
correct, total = 0, 0
all_probs, all_true = [], []

with torch.no_grad():
  for xb, yb in test_loader:
    xb, yb = xb.to(device), yb.to(device)
    logits = model(xb)
    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

correct += (preds == yb).sum().item()
total += yb.size(0)

all_probs.append(probs.cpu().numpy())
all_true.append(yb.cpu().numpy())

val_acc = correct / max(total, 1)
history.append({"epoch": epoch + 1, "train_loss": float(train_loss), "val_acc": float(val_acc)})
print(f"Epoch {epoch+1}/{epochs} | loss={train_loss:.4f} | val_acc={val_acc:.4f}")



Epoch 6/6 | loss=0.2857 | val_acc=0.9688


In [10]:
y_true = np.concatenate(all_true)
y_prob = np.concatenate(all_probs)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_true, y_pred)
try:
  roc = roc_auc_score(y_true, y_prob, multi_class="ovr")
except Exception:
    roc = 0.0
class_dist = {}
for idx, cls in enumerate(train_ds.classes):
  class_dist[cls] = int(sum(s[1] == idx for s in train_ds.samples))
meta = {
"accuracy": round(acc * 100, 2),
"roc_auc": round(roc * 100, 2),
"n_samples": len(train_ds) + len(test_ds),
"classes": train_ds.classes,
"class_distribution": class_dist,
"model_type": "efficientnet_b0",
"image_size": [224, 224],
"history": history,
"report": classification_report(y_true, y_pred, output_dict=True)
}

print("Final Accuracy:", meta["accuracy"])
print("Final ROC-AUC:", meta["roc_auc"])

Final Accuracy: 96.88
Final ROC-AUC: 0.0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np

num_classes = len(train_ds.classes)
labels = list(range(num_classes))

# predictions already computed:
# y_true, y_pred, y_prob

acc = accuracy_score(y_true, y_pred)

# robust classification report (no warnings spam)
report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=train_ds.classes,
    zero_division=0,
    output_dict=True
)

# robust multiclass ROC-AUC
y_true_bin = label_binarize(y_true, classes=labels)
per_class_auc = []
for i in range(num_classes):
    positives = y_true_bin[:, i].sum()
    negatives = len(y_true_bin[:, i]) - positives
    if positives > 0 and negatives > 0:
        per_class_auc.append(roc_auc_score(y_true_bin[:, i], y_prob[:, i]))

roc = float(np.mean(per_class_auc)) if per_class_auc else None

meta = {
    "accuracy": round(acc * 100, 2),
    "roc_auc": round(roc * 100, 2) if roc is not None else None,
    "n_samples": len(train_ds) + len(test_ds),
    "classes": train_ds.classes,
    "class_distribution": class_dist,
    "model_type": "efficientnet_b0",
    "image_size": [224, 224],
    "history": history,
    "report": report
}

In [15]:
print("Train classes:", train_ds.classes)
print("Test classes :", test_ds.classes)

Train classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Test classes : ['glioma', 'meningioma', 'notumor', 'pituitary']


In [18]:
from collections import Counter
import numpy as np

test_counts = Counter([y for _, y in test_ds.samples])
print("Test class counts:")
for i, c in enumerate(train_ds.classes):
    print(c, test_counts.get(i, 0))

print("Unique labels in y_true:", np.unique(y_true))
print("Shape y_prob:", y_prob.shape)

Test class counts:
glioma 400
meningioma 400
notumor 400
pituitary 400
Unique labels in y_true: [3]
Shape y_prob: (32, 4)


In [21]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

model.eval()
all_probs = []
all_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1).cpu().numpy()

        all_probs.append(probs)
        all_true.append(yb.numpy())

y_prob = np.concatenate(all_probs, axis=0)   # (N, 4)
y_true = np.concatenate(all_true, axis=0)    # (N,)
y_pred = np.argmax(y_prob, axis=1)

print("y_true unique:", np.unique(y_true))
print("y_prob shape :", y_prob.shape)

labels = list(range(len(train_ds.classes)))

acc = accuracy_score(y_true, y_pred)

report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=train_ds.classes,
    zero_division=0,
    output_dict=True
)

# robust multiclass ROC-AUC
y_true_bin = label_binarize(y_true, classes=labels)
valid_auc = []
for i in range(len(labels)):
    col = y_true_bin[:, i]
    if col.sum() > 0 and col.sum() < len(col):
        valid_auc.append(roc_auc_score(col, y_prob[:, i]))

roc_auc = float(np.mean(valid_auc)) if valid_auc else None

print("Accuracy:", round(acc * 100, 2))
print("ROC-AUC :", round(roc_auc * 100, 2) if roc_auc is not None else None)

y_true unique: [0 1 2 3]
y_prob shape : (1600, 4)
Accuracy: 91.62
ROC-AUC : 98.15


In [16]:
os.makedirs("brain_artifacts", exist_ok=True)

torch.save(
{
"state_dict": model.state_dict(),
"classes": train_ds.classes,
"image_size": [224, 224]
},
"brain_artifacts/brain_tumor_model.pt"
)
joblib.dump(train_ds.classes, "brain_artifacts/brain_tumor_labels.pkl")

with open("brain_artifacts/brain_tumor_meta.json", "w") as f:
  json.dump(meta, f)
print("Saved:")
print("- brain_artifacts/brain_tumor_model.pt")
print("- brain_artifacts/brain_tumor_labels.pkl")
print("- brain_artifacts/brain_tumor_meta.json")

Saved:
- brain_artifacts/brain_tumor_model.pt
- brain_artifacts/brain_tumor_labels.pkl
- brain_artifacts/brain_tumor_meta.json


In [17]:
from google.colab import files
files.download("brain_artifacts/brain_tumor_model.pt")
files.download("brain_artifacts/brain_tumor_labels.pkl")
files.download("brain_artifacts/brain_tumor_meta.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
import joblib, os
os.makedirs("brain_artifacts", exist_ok=True)
joblib.dump(train_ds.classes, "brain_artifacts/brain_tumor_labels.pkl")
print("saved:", train_ds.classes)

saved: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [23]:
from google.colab import files
files.download("brain_artifacts/brain_tumor_labels.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>